# Source-to-output framing study
Select GPU T4 x2 in Kaggle. Attach the study dataset or bundle. Read the guide in docs/kaggle_guide.md. The smoke test runs without a model.

In [ ]:
from pathlib import Path
import shutil, sys, subprocess, zipfile, os, json
INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
ROOT = Path(os.environ.get("FRAMING_STUDY_ROOT", WORK / "source_framing_study"))
RESTORE = None  # Optional path to a previous framing_results.zip
MODEL = "meta-llama/Llama-3.1-8B-Instruct"  # Or an attached HF checkpoint directory
REVISION = None  # Required for attached weights; use their version identifier
RUN_NAME = "pilot_fp16_2xt4"

def unpack(path, dest):
    with zipfile.ZipFile(path) as z:
        for item in z.namelist():
            if not (dest / item).resolve().is_relative_to(dest.resolve()):
                raise ValueError("Unsafe path in archive")
        z.extractall(dest)

if not (ROOT / "src/run.py").exists():
    folders = [p.parent.parent for p in INPUT.rglob("implementation_plan.md") if (p.parent.parent / "src/run.py").is_file()]
    if folders:
        shutil.copytree(folders[0], ROOT, dirs_exist_ok=True)
    else:
        bundles = list(INPUT.rglob("source_framing_kaggle.zip"))
        if len(bundles) != 1:
            raise RuntimeError("Attach exactly one study bundle or extracted dataset.")
        unpack(bundles[0], WORK)
if RESTORE:
    if (ROOT / "outputs/llama").exists():
        raise RuntimeError("Restore only into a fresh workspace to preserve existing annotations.")
    unpack(Path(RESTORE), ROOT)
os.chdir(ROOT)
def run(*args):
    subprocess.run([sys.executable, *map(str,args)], cwd=ROOT, check=True)
print(ROOT)

In [ ]:
run("src/prepare.py")
run("src/smoke.py")
print((ROOT / "outputs/dummy/fixture_v1/smoke_checks.json").read_text())

## Llama run
The next cells install GPU dependencies and run the real checkpoint. A mounted model can be used offline if the pinned packages or their wheels are already available.

In [ ]:
# Reuse Kaggle's CUDA PyTorch. Install study packages into a separate interpreter.
env_dir = Path("/kaggle/temp/source-framing-env")
study_python = env_dir / "bin/python"
if not study_python.exists():
    env_dir.parent.mkdir(parents=True, exist_ok=True)
    run("-m", "venv", "--system-site-packages", env_dir)
subprocess.run([str(study_python), "-m", "pip", "install", "-r",
                str(ROOT / "requirements-kaggle.txt")], check=True)
# Keep downloaded weights outside the exported results archive.
os.environ["HF_HOME"] = str(ROOT / ".model-cache")
if not Path(MODEL).is_dir() and not os.environ.get("HF_TOKEN"):
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
args = ["src/run.py","--mode","llama","--name",RUN_NAME,"--model",MODEL]
if REVISION:
    args.extend(["--revision",REVISION])
subprocess.run([str(study_python), *args], cwd=ROOT, check=True)

## Optional human results
After review, attach the three completed evaluation CSVs and set the directory below. Previously completed human sheets are preserved on a normal rerun.

In [ ]:
REVIEWED = None  # Example: Path("/kaggle/input/my-reviewed-sheets")
if REVIEWED:
    dest = ROOT / "data/annotations/llama" / RUN_NAME
    for role in ("reviewer_a","reviewer_b","adjudicated"):
        file = f"evaluation_{role}.csv"
        shutil.copyfile(Path(REVIEWED)/file, dest/file)
    run("src/analyze.py","--mode","llama","--name",RUN_NAME)

In [ ]:
from IPython.display import FileLink, display
archive = ROOT / "outputs/framing_results.zip"
with zipfile.ZipFile(archive,"w",zipfile.ZIP_DEFLATED) as z:
    for base in ("outputs/llama","data/annotations/llama","data/sources","docs"):
        for file in sorted((ROOT/base).rglob("*")):
            if file.is_file():
                z.write(file,file.relative_to(ROOT))
print(archive)
os.chdir(WORK)
display(FileLink(str(archive.relative_to(WORK))))